In [1]:
import pandas as pd
import numpy as np

# 1. Set the Random Seed
np.random.seed(42)

# Number of users we want to simulate
n_users = 5000

# 2. Generate Categorical Data (Subscription Tiers)
tiers = ['Basic', 'Pro', 'Enterprise']
subscription_tier = np.random.choice(tiers, size=n_users, p=[0.6, 0.3, 0.1])

# 3. Generate Numerical Data (Behavior & Friction)
support_tickets_opened = np.random.poisson(lam=1.2, size=n_users) 

avg_session_length_mins = np.random.normal(loc=25, scale=10, size=n_users)
avg_session_length_mins = np.clip(avg_session_length_mins, 1, 120) # Ensure no negative time!

# Generate days since last login (mostly between 1 and 30 days)
days_since_last_login = np.random.poisson(lam=10, size=n_users)


# 4. Engineering the "Churn" Logic
churn_risk_score = np.full(n_users, 0.10)

# Apply rules to increase or decrease the risk:
# Rule A: High days since login increases risk drastically
churn_risk_score += (days_since_last_login / 30) * 0.40 

# Rule B: High support tickets increase risk
churn_risk_score += (support_tickets_opened * 0.05)

# Rule C: Good session length decreases risk (they are using the app!)
churn_risk_score -= (avg_session_length_mins / 120) * 0.20

# Rule D: Enterprise users are less likely to churn (they sign contracts)
# We use boolean indexing here.
churn_risk_score[subscription_tier == 'Enterprise'] -= 0.15
churn_risk_score[subscription_tier == 'Pro'] -= 0.05

# Ensure probabilities stay strictly between 0% (0.0) and 100% (1.0)
churn_risk_score = np.clip(churn_risk_score, 0, 1)

# 5. Final Output: Did they churn? (1 = Yes, 0 = No)
churned = np.random.binomial(n=1, p=churn_risk_score)

# 6. Build the DataFrame and Save
df = pd.DataFrame({
    'user_id': range(1001, 1001 + n_users),
    'subscription_tier': subscription_tier,
    'days_since_last_login': days_since_last_login,
    'avg_session_length_mins': np.round(avg_session_length_mins, 1),
    'support_tickets_opened': support_tickets_opened,
    'churned': churned
})

# Save to your data folder
df.to_csv('saas_churn_data.csv', index=False)

print(f"Dataset generated successfully! Here is a preview:")
print(df.head())
print("\nChurn Rate Breakdown:")
print(df['churned'].value_counts(normalize=True) * 100)

Dataset generated successfully! Here is a preview:
   user_id subscription_tier  days_since_last_login  avg_session_length_mins  \
0     1001             Basic                      9                     22.9   
1     1002        Enterprise                     14                     19.9   
2     1003               Pro                      6                     17.3   
3     1004             Basic                     12                     16.2   
4     1005             Basic                     14                     18.5   

   support_tickets_opened  churned  
0                       1        0  
1                       1        0  
2                       1        0  
3                       2        0  
4                       1        0  

Churn Rate Breakdown:
churned
0    77.56
1    22.44
Name: proportion, dtype: float64
